# 01 — Data Collection

Collects daily **PETR4.SA** market data for the last five years from Yahoo Finance and saves the untransformed data to `data/raw`. Requires `pandas` and `yfinance`.

In [9]:
from pathlib import Path

import pandas as pd
import yfinance as yf

TICKER = "PETR4.SA"
PERIOD = "5y"
INTERVAL = "1d"

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "petr4_daily.csv"

In [10]:
df_raw = yf.Ticker(TICKER).history(
    period=PERIOD,
    interval=INTERVAL,
    auto_adjust=False,
    actions=True,
)

if df_raw.empty:
    raise RuntimeError(f"No data returned for {TICKER}.")

df_raw.index = pd.to_datetime(df_raw.index).tz_localize(None)
df_raw.index.name = "Date"
df_raw = df_raw.sort_index()
df_raw.head()

,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,
2021-07-21,26.790001,27.240000,26.590000,26.959999,7.760904,51116600,0.0,0.0
2021-07-22,27.000000,27.219999,26.709999,26.900000,7.743631,37736500,0.0,0.0
2021-07-23,27.100000,27.170000,26.680000,26.740000,7.697572,34025500,0.0,0.0
2021-07-26,26.740000,27.469999,26.719999,27.469999,7.907716,46802600,0.0,0.0
2021-07-27,27.350000,27.400000,26.900000,27.150000,7.815598,51901800,0.0,0.0


In [11]:
required_columns = {"Open", "High", "Low", "Close", "Volume"}
missing_columns = required_columns.difference(df_raw.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {sorted(missing_columns)}")
if not df_raw.index.is_unique:
    raise ValueError("Duplicate dates were found.")

pd.Series({
    "records": len(df_raw),
    "start_date": df_raw.index.min().date(),
    "end_date": df_raw.index.max().date(),
    "missing_values": int(df_raw.isna().sum().sum()),
})

records                 1248
start_date        2021-07-21
end_date          2026-07-21
missing_values             0
dtype: object

In [12]:
RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
df_raw.to_csv(RAW_PATH)